list vs set, two questions:

1. How much slower is `insert(0, x)` than `append(x)` on a list?
2. How much slower is `x in list` than `x in set`?

In [1]:
import timeit

SIZES = [10_000, 1_000_000]
REPEAT = 5


def per_op(stmt, setup, number):
    """Best time for one run of stmt, in microseconds."""
    times = timeit.repeat(stmt, setup=setup, number=number, repeat=REPEAT)
    return min(times) / number * 1e6


def fmt(us):
    return f"{us:12.4f} µs"

### 1. `append` vs. `insert(0, x)`

`append` - Amortized O(1) - adds to the end    
 `insert(0, x)` - O(n - k) - has to shift every element one slot to the right 

In [2]:
append_results = {}

for n in SIZES:
    setup = f"lst = list(range({n}))"
    t_append = per_op("lst.append(-1)", setup, number=1_000_000)
    t_insert = per_op("lst.insert(0, -1)", setup, number=1_000)
    append_results[n] = (t_append, t_insert)

    print(f"n = {n:>9,}")
    print(f"  append(x)    {fmt(t_append)}")
    print(f"  insert(0, x) {fmt(t_insert)}   ({t_insert / t_append:,.0f}x slower)")

n =    10,000
  append(x)          0.0336 µs
  insert(0, x)       2.3899 µs   (71x slower)


n = 1,000,000
  append(x)          0.0280 µs
  insert(0, x)     343.2463 µs   (12,246x slower)


In [3]:
small, big = SIZES
n_ratio = big / small
insert_ratio = append_results[big][1] / append_results[small][1]
append_ratio = append_results[big][0] / append_results[small][0]

print(f"items:          {n_ratio:>6,.0f}x more")
print(f"insert(0, x):   {insert_ratio:>6,.0f}x slower (O(n) predicts {n_ratio:,.0f}x)")
print(f"append(x):      {append_ratio:>6.1f}x slower (O(1) predicts 1x)")


items:             100x more
insert(0, x):      144x slower (O(n) predicts 100x)
append(x):         0.8x slower (O(1) predicts 1x)


### 2. `in` on a list vs. `in` on a set

A list checks items one by one, O(n). A set hashes the value and jumps straight to its slot, O(1) on average.

Three cases:

- the item is first (best case for the list)
- the item is last
- the item is missing (worst case for the list, it scans everything)

In [4]:
in_results = {}

for n in SIZES:
    setup = f"lst = list(range({n})); st = set(lst)"
    cases = {
        "first": "0",
        "last": str(n - 1),
        "missing": "-1",
    }
    print(f"n = {n:>9,}")
    for name, value in cases.items():
        # scanning a million items takes milliseconds, so run it fewer times
        list_number = 100_000 if name == "first" else max(10, 1_000_000 // n)
        t_list = per_op(f"{value} in lst", setup, number=list_number)
        t_set = per_op(f"{value} in st", setup, number=100_000)
        in_results[(n, name)] = (t_list, t_set)
        print(f"  {name:<8} list {fmt(t_list)}   set {fmt(t_set)}   ({t_list / t_set:,.0f}x)")

n =    10,000
  first    list       0.0231 µs   set       0.0292 µs   (1x)
  last     list      59.1530 µs   set       0.0203 µs   (2,920x)
  missing  list      55.1760 µs   set       0.0154 µs   (3,594x)
n = 1,000,000


  first    list       0.0158 µs   set       0.0131 µs   (1x)


  last     list   10682.9000 µs   set       0.0215 µs   (497,968x)


  missing  list   10646.5700 µs   set       0.0154 µs   (690,618x)


## Summary

In [5]:
small, big = SIZES
W = 24


def header(title):
    print()
    print(title)


def row(name, values):
    print(f"  {name:<{W}}{fmt(values[0]):>16}{fmt(values[1]):>16}")


def ratio(name, slow, fast):
    print(f"  {name:<{W}}" + "".join(f"{s / f:>15,.0f}x" for s, f in zip(slow, fast)))


print(f"{'':<{W + 2}}{f'n = {small:,}':>16}{f'n = {big:,}':>16}")
print("-" * (W + 34))

header("adding to a list")
append = [append_results[n][0] for n in SIZES]
insert = [append_results[n][1] for n in SIZES]
row("append(x)", append)
row("insert(0, x)", insert)
ratio("insert is slower by", insert, append)

for case in ("first", "last", "missing"):
    header(f"x in ..., item is {case}")
    lst = [in_results[(n, case)][0] for n in SIZES]
    st = [in_results[(n, case)][1] for n in SIZES]
    row("list", lst)
    row("set", st)
    ratio("list is slower by", lst, st)

                                n = 10,000   n = 1,000,000
----------------------------------------------------------

adding to a list
  append(x)                      0.0336 µs       0.0280 µs
  insert(0, x)                   2.3899 µs     343.2463 µs
  insert is slower by                  71x         12,246x

x in ..., item is first
  list                           0.0231 µs       0.0158 µs
  set                            0.0292 µs       0.0131 µs
  list is slower by                     1x              1x

x in ..., item is last
  list                          59.1530 µs   10682.9000 µs
  set                            0.0203 µs       0.0215 µs
  list is slower by                 2,920x        497,968x

x in ..., item is missing
  list                          55.1760 µs   10646.5700 µs
  set                            0.0154 µs       0.0154 µs
  list is slower by                 3,594x        690,618x
